# Pooled regions 1c-nt2p5b: fits, nt2p5b only (GPU session)

A copy of `Pooled Regions 1c - Fits (GPU session).ipynb` with `MODELS = ["nt2p5b"]`, so the five models can run in five GPU sessions at once (each writes its own `fits/<model>/`). Needs `fcav_pool/mpra_windows.json` from 1.5 of the main notebook. **GPU runtime.** Resumable: rerun from the top after a disconnect.

# Section 2 — embed and fit, five models *(GPU runtime)*

## 2.1 Settings, models and layers

The readout is the one of the cross-model notebooks: the mean hidden state over the tokens that touch the motif's
bases; for HyenaDNA (causal) also the reverse-complement pass, concatenated. NT-v2 500M with this readout is exactly
the legacy readout (checked there). `FIT_LAYERS` are the layers used before; they were chosen by held-out ChIP AUC,
not by the MPRA.

In [ ]:
!pip install -q -U "transformers==4.57.6" "tokenizers==0.22.0" "huggingface_hub==0.34.0" "safetensors==0.4.5"
!pip install -q scikit-learn scipy pandas
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
WORK_DIR = "/content/drive/MyDrive/GitHub/Biological-Foundation-Model/Notebooks/EP Pair Evaluation/per_occurrence/motif_token_study"
import os, json, time, gc, sys
import numpy as np
import torch
import torch.nn.functional as F
from contextlib import contextmanager
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from transformers import AutoTokenizer, AutoModel, AutoConfig, AutoModelForMaskedLM, AutoModelForCausalLM

C = json.load(open(f"{WORK_DIR}/config.json"))
POOL = f"{WORK_DIR}/fcav_pool"
MWJ = json.load(open(f"{POOL}/mpra_windows.json"))
BUILT = sorted(m for m, s in MWJ["status"].items() if s == "built")

REGISTRY = {
    "nt500m":   dict(hf="InstaDeepAI/nucleotide-transformer-v2-500m-multi-species", tok_map=("kmer", 6), causal=False, batch=32),
    "dnabert2": dict(hf="zhihan1996/DNABERT-2-117M",                                tok_map=("offsets",), causal=False, batch=32, out="dnabert2"),
    "genalm":   dict(hf="AIRI-Institute/gena-lm-bert-large-t2t",                    tok_map=("offsets",), causal=False, batch=32),
    "hyenadna": dict(hf="LongSafari/hyenadna-large-1m-seqlen-hf",                   tok_map=("char",), causal=True, batch=32, rc_mode="concat"),
    "nt2p5b":   dict(hf="InstaDeepAI/nucleotide-transformer-2.5b-multi-species",    tok_map=("kmer", 6), causal=False, batch=16),
}
MODELS = ["nt2p5b"]   # one model per GPU session; the other copies of 1c do the others
FIT_LAYERS = {"nt500m": [24], "dnabert2": [4], "genalm": [8, 11], "hyenadna": [3], "nt2p5b": [14]}   # the model's own layer ids
BASE_WEIGHTED = False     # equal weight to every token that touches the motif (= the legacy readout for NT)
BATCH = 32
WLEN, MOFF = C["window_len"], C["motif_offset"]
DEV = torch.device("cuda")
COMP = str.maketrans("ACGTN", "TGCAN")
rc = lambda s: s.translate(COMP)[::-1]

HBY = {}
for h in MWJ["hits"]:
    HBY.setdefault(h["motif"], []).append(h)
ORDER = sorted(BUILT, key=lambda m: (-len(HBY.get(m, [])), m))     # most hits first
print(f"{len(BUILT)} TFs with sites, {len(MWJ['hits'])} hit windows in {len(MWJ['regions'])} regions; models {MODELS}")

## 2.2 Tokens, models, readout (copied from the cross-model notebook)

The only change: `hidden_stack` keeps just the layers in `FIT_LAYERS`.

In [ ]:
def load_tokenizer(tag):
    spec = REGISTRY[tag]
    tokz = AutoTokenizer.from_pretrained(spec["hf"], trust_remote_code=True)
    tokz.padding_side = "right"          # several of these default to left (HyenaDNA)
    if tokz.pad_token is None:
        tokz.pad_token = tokz.eos_token if tokz.eos_token is not None else tokz.unk_token
    return tokz


def token_bases(spec, n):
    # (n, 2) base interval of each sequence token, for the arithmetic maps
    kind = spec["tok_map"][0]
    if kind == "kmer":
        k = spec["tok_map"][1]
        return np.stack([np.arange(n) * k, (np.arange(n) + 1) * k], 1)
    if kind == "char":
        return np.stack([np.arange(n), np.arange(n) + 1], 1)
    raise RuntimeError("offsets are resolved by the tokenizer, not here")


def encode(tokz, spec, seqs):
    # input_ids (B, T), attention mask (B, T), and per sequence: the positions of its sequence tokens in input_ids
    # and the base interval each one covers
    kind = spec["tok_map"][0]
    if kind == "offsets":
        assert tokz.is_fast, f"{spec['hf']} has no fast tokenizer, so offset mapping is unavailable"
        enc = tokz(seqs, return_tensors="pt", padding=True, return_offsets_mapping=True)
    else:
        enc = tokz(seqs, return_tensors="pt", padding=True)
    ids = enc["input_ids"]
    am = enc.get("attention_mask")
    if am is None:
        am = (ids != tokz.pad_token_id).long()
    amn = am.numpy()
    pos, spans = [], []
    if kind == "offsets":
        om = enc["offset_mapping"].numpy()
        for b in range(len(seqs)):
            keep = (om[b, :, 1] > om[b, :, 0]) & (amn[b] == 1)
            pos.append(np.flatnonzero(keep))
            spans.append(om[b, keep])
    else:
        for b in range(len(seqs)):
            sp = np.array(tokz.get_special_tokens_mask(ids[b].tolist(), already_has_special_tokens=True))
            p = np.flatnonzero((sp == 0) & (amn[b] == 1))
            pos.append(p)
            spans.append(token_bases(spec, len(p)))
        assert len({len(p) for p in pos}) == 1, f"{spec['hf']}: windows of equal length gave different token counts"
    return ids, am, pos, spans


def span_weights(bases, lo, hi):
    # per-token weight for the base interval [lo, hi)
    ov = np.minimum(bases[:, 1], hi) - np.maximum(bases[:, 0], lo)
    ov = np.clip(ov, 0, None).astype(np.float32)
    if not BASE_WEIGHTED:
        ov = (ov > 0).astype(np.float32)
    assert ov.sum() > 0, f"no token covers bases [{lo}, {hi})"
    return ov / ov.sum()


def load_model(tag):
    spec = REGISTRY[tag]
    # the class the checkpoint ships its own code for (NT-v2 has AutoModelForMaskedLM but no AutoModel)
    amap = getattr(AutoConfig.from_pretrained(spec["hf"], trust_remote_code=True), "auto_map", None) or {}
    cls = next((c for k, c in (("AutoModel", AutoModel), ("AutoModelForMaskedLM", AutoModelForMaskedLM),
                               ("AutoModelForCausalLM", AutoModelForCausalLM)) if k in amap), AutoModel)
    mdl = cls.from_pretrained(spec["hf"], trust_remote_code=True).to(DEV).eval()
    # DNABERT-2's triton flash-attention kernel does not compile on current triton; its code falls back to PyTorch
    for name in {type(m).__module__ for m in mdl.modules()}:
        mod = sys.modules.get(name)
        if mod is not None and getattr(mod, "flash_attn_qkvpacked_func", None) is not None:
            mod.flash_attn_qkvpacked_func = None
            print(f"{tag}: triton flash-attention disabled in {name.split('.')[-1]}; using PyTorch attention")
    if spec.get("out") == "dnabert2" and getattr(mdl, "pooler", None) is not None:
        mdl.pooler = None
    return mdl


POS, DEPTH, LOADED = {}, {}, {}


@torch.no_grad()
def hidden_stack(tag, model, ids, am):
    # one (layers, T_j, H) float tensor per sequence; only the layers in POS[tag] once prepare() has set it
    spec = REGISTRY[tag]
    if spec.get("out") == "dnabert2":
        # DNABERT-2 returns every encoder layer unpadded; one sequence at a time keeps the token order
        out = []
        for j in range(ids.shape[0]):
            t = ids[j][am[j].bool()][None].to(DEV)
            layers, _ = model(t, attention_mask=torch.ones_like(t), output_all_encoded_layers=True)
            S = torch.stack([x.reshape(t.shape[1], -1) for x in layers], 0).float()
            out.append(S[POS[tag]] if tag in POS else S)
        return out
    kw = {} if spec["causal"] else {"attention_mask": am.to(DEV)}
    hs = model(input_ids=ids.to(DEV), output_hidden_states=True, **kw).hidden_states
    if tag in POS:
        hs = [hs[i] for i in POS[tag]]
    S = torch.stack(hs, 1).float()
    return [S[j] for j in range(S.shape[0])]


PROBE = [h["window"] for h in MWJ["hits"][:4]]


def prepare(tag):
    # load the model; from one forward pass: number of layers, token tiling, and where FIT_LAYERS sit in the stack
    spec = REGISTRY[tag]
    mdl = load_model(tag)
    tokz = LOADED.get(tag, {}).get("tokz") or load_tokenizer(tag)
    LOADED[tag] = dict(spec=spec, tokz=tokz)
    POS.pop(tag, None)
    ids, am, pos, spans = encode(tokz, spec, PROBE[:2])
    S = hidden_stack(tag, mdl, ids, am)
    for j, Sj in enumerate(S):
        assert Sj.ndim == 3 and int(pos[j].max()) < Sj.shape[1]
        b = spans[j]
        assert b[0, 0] == 0 and b[-1, 1] == WLEN and (b[1:, 0] == b[:-1, 1]).all(), f"{tag}: tokens do not tile"
    nl = S[0].shape[0]
    lids = list(range(1, nl + 1)) if spec.get("out") == "dnabert2" else list(range(nl))   # DNABERT-2: no embedding layer
    DEPTH[tag] = lids[-1]
    miss = [l for l in FIT_LAYERS[tag] if l not in lids]
    assert not miss, f"{tag}: layers {miss} do not exist (ids {lids[0]}..{lids[-1]})"
    POS[tag] = [lids.index(l) for l in FIT_LAYERS[tag]]
    print(f"{tag}: {nl} layers (depth {DEPTH[tag]}), fitting {FIT_LAYERS[tag]}, hidden {S[0].shape[2]}, "
          f"{len(pos[0])} tokens for {WLEN} bp")
    del S
    return mdl


@torch.no_grad()
def readout(tag, model, seqs, lohi):
    # (n, layers, D): D = hidden, or 2 * hidden for a causal model (forward ‖ reverse complement)
    L = LOADED[tag]
    tokz, spec = L["tokz"], L["spec"]
    passes = [(seqs, lohi)]
    if spec["causal"]:
        passes.append(([rc(s) for s in seqs], [(WLEN - b, WLEN - a) for a, b in lohi]))
    bs = spec.get("batch", BATCH)
    parts = []
    for ss, lh in passes:
        out = None
        for i in range(0, len(ss), bs):
            chunk, lhc = ss[i:i + bs], lh[i:i + bs]
            ids, am, pos, spans = encode(tokz, spec, chunk)
            S = hidden_stack(tag, model, ids, am)
            if out is None:
                out = np.zeros((len(ss), S[0].shape[0], S[0].shape[2]), np.float16)
            for j in range(len(chunk)):
                w = torch.tensor(span_weights(spans[j], *lhc[j]), device=DEV)
                p = torch.tensor(pos[j], device=DEV)
                out[i + j] = (S[j][:, p] * w[None, :, None]).sum(1).half().cpu().numpy()
            del S
        parts.append(out)
    if len(parts) == 1:
        return parts[0]
    if spec.get("rc_mode", "concat") == "mean":
        return ((parts[0].astype(np.float32) + parts[1].astype(np.float32)) / 2).astype(np.float16)
    return np.concatenate(parts, 2)

## 2.3 The estimator (copied unchanged from the cross-model notebook)

In [ ]:
torch.backends.cuda.preferred_linalg_library("cusolver")
LAMS = torch.logspace(4, -1, 11, device=DEV)      # L2 penalties tried, strongest first (standardised features, summed log-loss)
NEWTON = 15                                       # max Newton steps per penalty
INNER = 3                                         # inner CV folds for the penalty
FAST = True       # Hessian in TF32 / Cholesky in float32, float64 fallback for the layers where that fails


@contextmanager
def _tf32(on):
    old = torch.backends.cuda.matmul.allow_tf32
    torch.backends.cuda.matmul.allow_tf32 = on
    try:
        yield
    finally:
        torch.backends.cuda.matmul.allow_tf32 = old


def logit_path(X, y):
    # L2 logistic regression for every layer at once and every penalty in LAMS (warm-started, strongest first).
    # X: (layers, n, d) standardised; y: (n,) 0/1. Returns W (lams, layers, d + 1) = [weights, intercept].
    nl, n, d = X.shape
    Xa = torch.cat([X, torch.ones(nl, n, 1, device=X.device)], 2)
    pen = torch.ones(d + 1, device=X.device); pen[-1] = 0                # intercept not penalised
    w = torch.zeros(nl, d + 1, device=X.device)
    w[:, -1] = torch.logit(y.mean().clamp(1e-3, 1 - 1e-3))

    def loss(w, lam):
        z = torch.einsum("lnd,ld->ln", Xa, w)
        return F.binary_cross_entropy_with_logits(z, y.expand_as(z), reduction="none").sum(1) + lam / 2 * (pen * w * w).sum(1)

    out = []
    for lam in LAMS.tolist():
        for _ in range(NEWTON):
            z = torch.einsum("lnd,ld->ln", Xa, w)
            p = torch.sigmoid(z)
            g = torch.einsum("lnd,ln->ld", Xa, p - y) + lam * pen * w
            s_ = p * (1 - p)
            with _tf32(FAST):
                H = (Xa * s_[..., None]).transpose(1, 2) @ Xa + torch.diag(lam * pen + 1e-6)
            if FAST:
                Lc, info = torch.linalg.cholesky_ex(H)
                step = torch.cholesky_solve(g[..., None], Lc)[..., 0]
                bad = (info > 0) | ~torch.isfinite(step).all(1)
                if bad.any():                                                 # exact Hessian, float64, those layers only
                    Xb = Xa[bad].double()
                    Hb = (Xb * s_[bad].double()[..., None]).transpose(1, 2) @ Xb + torch.diag(lam * pen + 1e-6).double()
                    Lb, ib = torch.linalg.cholesky_ex(Hb)
                    sb = torch.cholesky_solve(g[bad].double()[..., None], Lb)[..., 0].float()
                    sb[ib > 0] = 0
                    step[bad] = sb
                    del Xb, Hb, Lb
            else:
                Lc, info = torch.linalg.cholesky_ex(H.double())
                step = torch.cholesky_solve(g.double()[..., None], Lc)[..., 0].float()
                step[info > 0] = 0
            step = torch.where(torch.isfinite(step), step, torch.zeros_like(step))
            ts = torch.tensor([1, .5, .25, .125, 0], device=X.device)
            L = torch.stack([loss(w - t * step, lam) for t in ts])         # (steps, layers); t = 0 keeps w
            L = torch.nan_to_num(L, nan=float("inf"))
            t = ts[L.argmin(0)][:, None]
            w = w - t * step
            if (t * step).abs().max() < 1e-4:
                break
        out.append(w.clone())
    return torch.stack(out)


def bce(Xte, yte, W):
    # held-out summed log-loss of every (penalty, layer); Xte (layers, n, d), W (lams, layers, d + 1)
    z = torch.einsum("lnd,kld->kln", Xte, W[..., :-1]) + W[..., -1:]
    return F.binary_cross_entropy_with_logits(z, yte.expand_as(z), reduction="none").sum(2)


def fit_logit(X, y, seed):
    # penalty per layer by INNER-fold CV log-loss on these sites, then the model on all of them at that penalty
    ll = torch.zeros(len(LAMS), X.shape[0], device=X.device)
    yn = y.cpu().numpy()
    for itr, ite in StratifiedKFold(INNER, shuffle=True, random_state=seed).split(np.zeros(len(yn)), yn):
        itr, ite = torch.tensor(itr, device=X.device), torch.tensor(ite, device=X.device)
        ll += bce(X[:, ite], y[ite], logit_path(X[:, itr], y[itr]))
    k = torch.nan_to_num(ll, nan=float("inf")).argmin(0)
    Wp = logit_path(X, y)
    return Wp[k, torch.arange(X.shape[0])], k.cpu().numpy(), Wp


def fit_one(X, y, seed, Hh):
    # 5 outer folds, penalty by inner CV; out-of-fold AUC per layer; each hit's logit averaged over the 5 fold models
    folds = list(StratifiedKFold(5, shuffle=True, random_state=seed).split(X, y))
    Xg = torch.tensor(X, dtype=torch.float32, device=DEV).transpose(0, 1).contiguous()
    yg = torch.tensor(y, dtype=torch.float32, device=DEV)
    Hg = torch.tensor(Hh, dtype=torch.float32, device=DEV).transpose(0, 1).contiguous()
    nl = Xg.shape[0]
    oof = np.zeros((len(y), nl), np.float32)
    dk, qk, kk = [], [], []
    for f, (tr, te) in enumerate(folds):
        mu, sd = Xg[:, tr].mean(1, keepdim=True), Xg[:, tr].std(1, keepdim=True) + 1e-6
        W, ki, _ = fit_logit((Xg[:, tr] - mu) / sd, yg[tr], seed + f)
        kk.append(ki)
        oof[te] = (torch.einsum("lnd,ld->ln", (Xg[:, te] - mu) / sd, W[:, :-1]) + W[:, -1:]).cpu().numpy().T
        dh = (torch.einsum("lnd,ld->ln", (Hg - mu) / sd, W[:, :-1]) + W[:, -1:]).cpu().numpy().T
        dk.append(dh); qk.append(1 / (1 + np.exp(-dh)))
    assert np.isfinite(oof).all(), "non-finite out-of-fold logits"
    auc = np.array([roc_auc_score(y, oof[:, li]) for li in range(nl)])
    mu, sd = Xg.mean(1, keepdim=True), Xg.std(1, keepdim=True) + 1e-6
    Wp = logit_path((Xg - mu) / sd, yg)
    kall = np.median(np.array(kk), 0).round().astype(int)
    W = Wp[torch.tensor(kall, device=DEV), torch.arange(nl, device=DEV)]
    res = dict(auc=auc, oof=oof, cls=y, w_all=W.cpu().numpy(), mu=mu[:, 0].cpu().numpy(), sd=sd[:, 0].cpu().numpy(),
               lam_all=LAMS.cpu().numpy()[kall], dec_hit=np.mean(dk, 0).astype(np.float32),
               prob_hit=np.mean(qk, 0).astype(np.float32))
    del Xg, Hg, Wp; gc.collect(); torch.cuda.empty_cache()
    return res

## 2.4 Fit every TF on every model

One model at a time, TFs with the most hits first. Each `fits/<model>/<TF>.npz` holds the out-of-fold AUC, every hit's
P(bound) (`prob_hit`, `dec_hit` = its logit) and the hits' `(hit_region, hit_id)`. Written to a temporary name and
renamed, so an interrupted write never leaves a broken file. The printed "h left" is a running estimate.

In [ ]:
t_all = time.time()
for tag in MODELS:
    OUTD = f"{POOL}/fits/{tag}"
    os.makedirs(OUTD, exist_ok=True)
    todo = [m for m in ORDER if not os.path.exists(f"{OUTD}/{m}.npz")]
    print(f"\n=== {tag}: {len(todo)} of {len(ORDER)} TFs to do", flush=True)
    if not todo:
        continue
    mdl = prepare(tag)
    t_m = time.time()
    for k, m in enumerate(todo):
        t0 = time.time()
        J = json.load(open(f"{POOL}/sites/{m}.json"))
        R = J["records"]
        w = int(J.get("width", (J["K1"] - J["K0"]) * 6))
        y = np.array([r["cls"] for r in R])
        hi = HBY[m]
        assert all(h["end"] - h["start"] == w for h in hi)
        X = readout(tag, mdl, [r["window"] for r in R], [(MOFF, MOFF + w)] * len(R))
        XH = readout(tag, mdl, [h["window"] for h in hi], [(MOFF, MOFF + w)] * len(hi))
        assert np.isfinite(X).all() and np.isfinite(XH).all(), f"{tag} {m}: a hidden state overflowed float16"
        res = fit_one(X, y, C["seed"], XH)
        tmp = f"{OUTD}/{m}.part.npz"
        np.savez(tmp, hit_region=np.array([h["region"] for h in hi]), hit_id=np.array([h["hit_id"] for h in hi]),
                 layers=np.array(FIT_LAYERS[tag]), n_layers=DEPTH[tag], model=tag, rc=REGISTRY[tag]["causal"],
                 rc_mode=REGISTRY[tag].get("rc_mode", "concat"), base_weighted=BASE_WEIGHTED, **res)
        os.replace(tmp, f"{OUTD}/{m}.npz")
        left = (time.time() - t_m) / (k + 1) * (len(todo) - k - 1) / 3600
        print(f"[{tag} {k + 1}/{len(todo)}] {m:10s} sites {len(y):5d}  hits {len(hi):3d} in "
              f"{len({h['region'] for h in hi}):2d} regions  AUC " + "  ".join(f"L{l} {a:.3f}" for l, a in zip(FIT_LAYERS[tag], res["auc"]))
              + f"  [{time.time() - t0:.0f}s; ~{left:.1f} h left for {tag}]", flush=True)
        del X, XH; gc.collect(); torch.cuda.empty_cache()
    del mdl; gc.collect(); torch.cuda.empty_cache()
print(f"\ndone [{(time.time() - t_all) / 60:.1f} min]")

In [ ]:
from google.colab import runtime
runtime.unassign()